# Validation after transformation to see if everything works as planned

In [0]:
df = spark.table("marathos.silver.races_clean")

print(df.columns)

In [0]:
total = df.count()
unique = df.distinct().count()

print(f"Total: {total}")
print(f"Unique rows: {unique}")

In [0]:
df.select("year_of_event", "event_dates", "athlete_performance").limit(5).display()

#### Null values left

In [0]:
from pyspark.sql.functions import col
df.filter(col("event_distance_length").isNull()).show()
df.filter(col("athlete_performance").isNull()).show()

#### Removal of metrics ending with wrong value

In [0]:
df.filter(col("event_distance_length").like("%d%")).show()
df.filter(col("athlete_performance").like("%d%")).show()
df.filter(col("event_distance_length").like("%k")).show()
df.filter(col("athlete_performance").like("%k")).show()

In [0]:
df.filter(col("athlete_country").like("XXX")).show()

In [0]:
df.select("event_dates").show(10)

In [0]:
df.filter(col("event_dates").isNull()).count()

#### Date transformation did not work as intended

In [0]:
df_bronze = spark.table("marathos.bronze.races")
df_bronze.select("`Event dates`").distinct().show(20, truncate=False)

#### The following is regex headache

In [0]:
from pyspark.sql.functions import regexp_extract

df_bronze.select(
    regexp_extract("`Event dates`", r"^(\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4})$", 1).alias("cross_month")
).filter(col("cross_month") != "").distinct().count()

In [0]:
from pyspark.sql.functions import regexp_replace

df_bronze.select(
    regexp_replace("`Event dates`", r"(\d{2}\.\d{2}\.)-\d{2}\.\d{2}\.(\d{4})", "$1$2").alias("fixed")
).filter(col("fixed").like("30.04%")).show(5)

In [0]:
df_bronze.select(
    regexp_replace(
        regexp_replace("`Event dates`", r"(\d{2})\.-(\d{2})\.", "$1."),
        r"(\d{2}\.\d{2}\.)-\d{2}\.\d{2}\.(\d{4})", "$1$2"
    ).alias("fixed")
).filter(~col("fixed").rlike(r"^\d{2}\.\d{2}\.\d{4}$")).distinct().show(20, truncate=False)

In [0]:
df_bronze.select("`Event dates`").filter(
    col("`Event dates`").rlike(r"\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}")
    | col("`Event dates`").rlike(r"\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}")
).count()

In [0]:
df_bronze.select(
    regexp_replace("`Event dates`", r"(\d{2}\.\d{2}\.)-\d{2}\.\d{2}\.(\d{4})", "$1$2").alias("fixed")
).filter(col("fixed").like("19.06%")).show(5)

In [0]:
df_bronze.select(
    regexp_replace("`Event dates`", r"(\d{2}\.\d{2}\.\d{4})-\d{2}\.\d{2}\.\d{4}", "$1").alias("fixed")
).filter(col("fixed").like("19.06%")).show(5)

In [0]:
# Show distinct values for each column
for column in df.columns:
    distinct_count = df.select(column).distinct().display()

In [0]:
df.filter(col("athlete_performance").contains("Etap")).display()

- Countries to uppercase
- Drop weird average speed
- Athlete performance, what does it even mean?
- Replace X from gender to Other - check (null?)
- M = Male, F = Female - check
- Drop events from event_dates that doesn't match with year_of_event - check
- Drop dates like 6h,5h,4h - check
- Miles/mile to mi - check
- Drop /Etappen/Etap - check
- Drop those with missing metric value
- Drop athlete year of birth 1193 - check
- Drop age under 18 years old -check
- Event name, remove ""/''?, Country code?, length?
- Event distance/length, convert 4x52 km? split h/mi/km to separete columns? convert mi to km for simplicity?
- Athlete club, null to "unknown". Do the same for other columns?
- Athlete country - ingest proper country code dataset and replace
